In [ ]:
## IMPORTS AND SETUP
# Jupyter Notebook setup
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
sys.path.insert(0, "/tf/projet") # Add the project root directory to the Python path (docker hosting)
import tensorflow as tf
import wandb
import datetime
from wandb.integration.keras import WandbMetricsLogger
from dotenv import load_dotenv
from Leyanda_Project.utils.warning_clean import silence_tensorflow_warnings
from Leyanda_Project.models.callbacks import create_callbacks
from Leyanda_Project.models.cnn_classifier import create_model, train_model
from Leyanda_Project.preprocessing.binary_converter import convert_to_binary_dataset_structure
from Leyanda_Project.preprocessing.data_format import data_formats_fixes
from Leyanda_Project.preprocessing.data_loader import dataset_assembly, dataset_split
from Leyanda_Project.utils.naming import generate_model_name, get_model_path
from Leyanda_Project.utils.visualization import visualize_class_samples, visualize_class_distribution
from Leyanda_Project.utils.evaluation import make_inference, generate_confusion_matrices

# Suppress warnings
silence_tensorflow_warnings()

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU is available: {len(gpus)} device(s) detected")
    except RuntimeError as e:
        print("Error configuring GPU:", str(e))
else:
    print("No GPU available, using CPU")

# Wandb setup
if not os.path.exists("/tf/projet/.env"):
    print("WARNING: No .env file found, please create one with your Wandb API key.")
    exit(1)
else:
    load_dotenv("/tf/projet/.env")
    WANDB_API_KEY = os.getenv("API_KEY")
    wandb_entity = "tom-antoine-cesi"

In [ ]:
## PARAMETERS
seed = 123
project_name = "Leyanda"
model_arch = "CNN"

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset" # Path to the raw data folder
data_format_fix = False # Set to True to fix data formats
data_visualization = True # Set to True to visualize data
excluded_data_folders = ["Dataset Livrable 2"] # Folders to exclude from the dataset
batch_size = 64 # Batch size for dataset loading
img_height = 180 # Image height for dataset loading
img_width = 180 # Image width for dataset loading
image_size = f"{img_height}x{img_width}"

# Dataset split parameters
train_split = 0.8 # Proportion of the dataset to use for training
val_split = 0.1 # Proportion of the dataset to use for validation
test_split = 0.1 # Proportion of the dataset to use for testing

# Model configuration parameters
learning_rate = 0.001
epochs = 10
transfer_learning = False # Set to True to create a VGG16 model with custom top, False to create a custom model
train_transfer_model = False # Set to True to train the VGG-16 model
class_weight = False # Set to True to use class weights for imbalanced datasets
target_binary_class_name = "Photo" # Name of the target binary class for binary classification ("Photo", "Painting"...), None for multi-class
save_path = "/tf/projet/Leyanda_Project/models/saved" # Path to the models folder
binary_dataset_folder_output = f"/tf/projet/Dataset_binary_{target_binary_class_name}" # Path to the binary dataset folder
inference_image_path = "/tf/projet/Dataset/Photo/photo_0001.jpg" # Path to the image for inference

In [ ]:
## MODEL NAME AND PATH GENERATION
# Generate model name
model_name = generate_model_name(
    project_name=project_name,
    model_arch=model_arch,
    target_class=target_binary_class_name if target_binary_class_name else None,
    transfer_learning=transfer_learning,
    epoch=epochs,
    batch_size=batch_size,
    class_weight=class_weight,
    train_transfer_model=train_transfer_model
)

model_path = get_model_path(
    model_name=model_name,
    models_folder=save_path,
)
print(f"Model will be saved to: {model_path}")

In [ ]:
## DATA PREPARATION WORKFLOW
# Optional: Convert dataset to binary structure
if target_binary_class_name is not None:
    raw_data_path = convert_to_binary_dataset_structure(
        source_path=raw_data_path,
        target_binary_class=target_binary_class_name,
        output_path=binary_dataset_folder_output
    )


# 1. Check dataset existence
print(f"--Checking directory: {raw_data_path}--")
if not os.path.isdir(raw_data_path):
    print(f"Directory {raw_data_path} doesn't exist")
    exit(1)
else:
    print(f"Directory {raw_data_path} exists")


# Optional: Fix data formats
if data_format_fix:
    data_formats_fixes(raw_data_path=raw_data_path)
subfolders = [f for f in os.listdir(raw_data_path) if os.path.isdir(os.path.join(raw_data_path, f)) and f not in excluded_data_folders]


# 2. Assemble dataset
dataset, class_names = dataset_assembly(
    raw_data_path=raw_data_path,
    subfolders=subfolders,
    batch_size=batch_size,
    img_height=img_height,
    img_width=img_width,
    seed=seed
)


# 3. Split dataset
train_ds, val_ds, test_ds = dataset_split(
    dataset=dataset,
    train_split=train_split,
    val_split=val_split,
    batch_size=batch_size,
    seed=seed
)

# Optional: data visualization
if data_visualization:
    visualize_class_samples(dataset=train_ds, class_names=class_names, samples_per_class=5)
    visualize_class_distribution(dataset=train_ds, class_names=class_names)

In [ ]:
## MODELS WORKFLOW
# 1. Create model
model = create_model(
    model_name=model_name,
    input_shape=(img_height, img_width, 3),
    class_names=class_names,
    train_transfer_model=train_transfer_model,
    transfer_learning=transfer_learning,
    target_binary_class_name=target_binary_class_name
)

# 2. Initialize callbacks
run = wandb.init(
    entity=wandb_entity,
    project=project_name,
    name=f"{model_name}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}",
    reinit=True,
    config={
        "model": model_name,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "binary": target_binary_class_name,
    }
)
callback = [WandbMetricsLogger()] + create_callbacks(
    model_name=model_name,
    tensorboard=True,
    early_stopping=True,
    model_checkpoint=True,
    conf_matrix=True,
    val_data=test_ds,
    class_names=class_names
)

# 3. Train model
train_model(
    model=model,
    model_name=model_name,
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=epochs,
    save_path=save_path,
    class_weight=class_weight,
    target_binary_class_name=target_binary_class_name,
    callbacks=callback,
    wandb=wandb
)

print(f"Run summary : {run.summary}")
run.finish()

In [ ]:
## EVALUATION AND INFERENCE
# 1. Make inference
make_inference(
    model=model,
    class_names=class_names,
    target_binary_class_name=target_binary_class_name,
    img_path=inference_image_path)

# 2. Generate confusion matrices for all stored models (TODO: port to wandb)
# generate_confusion_matrices(test_ds=test_ds)